In [2]:
import anthropic
import json
import pandas as pd

client = anthropic.Anthropic()

def match_mine_names(list1, list2):
    prompt = f"""
    You are an expert in mining industry. Below are two lists of iron ore mine operator names 
    from two different datasets. Many refer to the same mine but with different naming conventions.
    
    Match each name in List 1 to the most likely same mine in List 2.
    Return a JSON array like:
    [{{"list1": "name", "list2": "name", "confidence": 0.95, "reason": "brief reason"}}]
    Only return JSON, no other text.
    
    List 1:
    {json.dumps(list1, indent=2)}
    
    List 2:
    {json.dumps(list2, indent=2)}
    """
    
    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=4000,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return json.loads(response.content[0].text)

iron_mines = pd.read_excel('../data/Processed_data/iron_mines_data.xlsx', index_col=False)
# Read starting from the 13th row of the "Data" sheet
cost_data = pd.read_excel('../data/S&PGlobal iron ore mining data/SPGlobal_MineEconomics_22-Apr-2026.xlsx', sheet_name='Data', skiprows=12, index_col=False)
df1_names = iron_mines['Asset name (English)'].dropna().tolist()
df2_names = cost_data['Property'].dropna().tolist()
# matches = match_mine_names(df1_names, df2_names)
# print(json.dumps(matches, indent=2))

print(df2_names)

# Given no API key, this function cannot run. Output df1_names and df2_names to Cloude and do the match there

['Taihang', 'Karnataka', 'Mikhailovsky GOK', 'Karnataka', 'Mikhailovsky GOK', 'Stoylensky GOK', 'Benxi City', 'Sokolov-Sarbai', 'Stoylensky GOK', 'Stoylensky GOK', 'Gudai-Darri', 'Gudai-Darri', 'Kachkanarsky GOK', 'Noamundi', 'Robe River (Pannawonica)', 'Chattisgarh Group', 'Robe River (Pannawonica)', 'Noamundi', 'Donimalai', 'Chattisgarh Group', 'Chichester Hub', 'Solomon Hub', 'Donimalai', 'Vargem Grande', 'Vargem Grande', 'Kachkanarsky GOK', 'Yandicoogina', 'Yandi', 'Yandi', 'Sokolov-Sarbai', 'Western Hub', 'Mt Tom Price', 'Minas Centrais', 'Mariana', 'Itabira', 'Minas Centrais', 'Mt Tom Price', 'Mariana', 'Channar', 'Tiecheng', 'Panzhihua Steel Zhujiabaobao', "Manyin'gou", 'Jimblebar', 'Channar', 'Roy Hill', 'Roy Hill', 'Huanian', 'Northern System', 'Red Hill', 'Silvergrass', 'Newman', 'Silvergrass', 'Jimblebar', 'Itabira', 'Shujigou', 'Newman', 'Samarco', 'Xinsheng', 'Northern System', 'Liyu', 'Serra Azul', 'West Angelas', 'West Angelas', 'Dalizi', 'Serra Azul', 'Casa de Pedra', '

In [3]:
# Cloude output the matching results and save in ../data/Processed_data/mine_asset_matches_v4.xlsx', index_col=False)
# Match mine names to operator
global_tracker_data = pd.read_excel('../data/Processed_data/Iron_mines_data_w_cost.xlsx', index_col=False)
sp_cost = pd.read_excel('../data/Processed_data/mine_asset_matches_v4.xlsx', index_col=False)
# Based on the column 'Best Match Asset Name (List 1)' in sp_cost, find the corresponding operator name in the iron_mines dataframe, and add a new column 'Operator' in sp_cost with the operator name. If there is no match, leave the cell value blank.
sp_cost = sp_cost.merge(global_tracker_data[['Asset name (English)', 'Operator']], left_on='Best Match Asset Name (List 1)', right_on='Asset name (English)', how='left')
sp_cost = sp_cost.drop(columns=['Asset name (English)'])
# sp_cost.to_excel('../data/Processed_data/mine_asset_matches_v5.xlsx', index=False)

In [ ]:
# Check if the names in S&P global data exist in the global tracker data
# # Read the data in mine_asset_matches_final.xlsx from row 195 but with column names kept
# mine_asset_matches = pd.read_excel('../data/Processed_data/mine_asset_matches_final.xlsx', skiprows=193, index_col=False)
# # Add column names to mine_asset_matches from the first row of the original excel file
# original = pd.read_excel('../data/Processed_data/mine_asset_matches_final.xlsx', index_col=False)
# mine_asset_matches.columns = original.columns

# # Match the name in the second column of mine_asset_matches with all columns of global_tracker_data, and for each name, count how many times it appears in the global_tracker_data, and add the count number as a new column 'Count in Global Tracker' in mine_asset_matches. If there is no match, the count number should be 0.
# # original_global_tracker = pd.read_excel('../data/GlobalTrackerData/iron/Global-Energy-Ownership-Tracker-February-2026-V1.xlsx', index_col=False)
# def count_matches(name, df):
#     count = 0
#     for col in df.columns:
#         count += df[col].astype(str).str.contains(name, case=False, na=False).sum()
#     return count
# mine_asset_matches['Count in Global Tracker'] = mine_asset_matches['Mine Short Name (List 2)'].apply(lambda x: count_matches(x, global_tracker_data) if pd.notna(x) else 0)
# # mine_asset_matches.to_excel('../data/Processed_data/mine_asset_matches_final_with_counts.# xlsx', index=False)      



In [27]:
# Left join global_tracker_data to S&P global cost data based on the matched mine names and operators, and keep all rows in global_tracker_data. If there is no match, leave it blank.
sp_cost_w_mineName = pd.read_excel('../data/S&PGlobal iron ore mining data/SPGlobal_MineEconomics_22-Apr-2026.xlsx', sheet_name='Data', skiprows=12, index_col=False)
# Only join the column 'Product', 'Fe Content (%)', ' Total Cash Cost (FOB) ($/dmt)' and 'Seaborne Shipment ($/dmt)'
sp_cost_w_mineName = sp_cost_w_mineName[['Property', 'Product', 'Fe Content (%)', ' Total Cash Cost (FOB) ($/dmt)', ' Seaborne Shipment ($/dmt)', 'Correct_operator', 'Correct_mine']]
merged_df = global_tracker_data.merge(sp_cost_w_mineName, left_on=['Asset name (English)', 'Operator'], right_on=['Correct_mine', 'Correct_operator'], how='left').drop(columns=['Property','Correct_mine', 'Correct_operator', ' Seaborne Shipment ($/dmt)'])
merged_df.rename(columns={' Total Cash Cost (FOB) ($/dmt)': 'Cost of Material ($/t)'}, inplace=True)
merged_df.head()


,GEM Asset ID,Asset name (English),Asset name (other language),Coordinates,Lat,Lon,City,State/Province,Country,Region,...,Primary Owner Country,Secondary Owner Country,Third Owner Country,Fourth Owner Country,Fifth Owner Country,Utility factor_2024,GEM wiki page URL,Product,Fe Content (%),Cost of Material ($/t)
0,P100000129290,Ghoryan Mine,NaN,"34.012090, 61.564895",34.012090,61.564895,unknown,Herat,Afghanistan,Asia Pacific,...,NaN,NaN,NaN,NaN,NaN,0.709983,https://www.gem.wiki/Ghoryan_Mine,NaN,NaN,NaN
1,P100000128013,Hajigak Mine,NaN,"34.667981, 68.062676",34.667981,68.062676,Hajigak,Bamyan,Afghanistan,Asia Pacific,...,NaN,NaN,NaN,NaN,NaN,0.709983,https://www.gem.wiki/Hajigak_Mine,NaN,NaN,NaN
2,P100000129288,Herat Mine,NaN,"34.123862, 61.681827",34.123862,61.681827,unknown,Herat,Afghanistan,Asia Pacific,...,NaN,NaN,NaN,NaN,NaN,0.709983,https://www.gem.wiki/Herat_Mine,NaN,NaN,NaN
3,P100000129289,Syadara Mine,NaN,"34.619615, 66.896422",34.619615,66.896422,unknown,unknown,Afghanistan,Asia Pacific,...,NaN,NaN,NaN,NaN,NaN,0.709983,https://www.gem.wiki/Syadara_Mine,NaN,NaN,NaN
4,P100000128018,FERAAL Ghara Djebilet Mine,NaN,"26.766298, -7.333398",26.766298,-7.333398,Tindouf,Tindouf,Algeria,Africa,...,Algeria,NaN,NaN,NaN,NaN,0.041667,https://www.gem.wiki/FERAAL_Ghara_Djebilet_Mine,NaN,NaN,NaN


In [28]:
merged_df.to_excel('../data/Processed_data/Iron_mines_data_w_cost_v2.xlsx', index=False)